# Experiment 3: physics-only vs data-assisted PINN
Compare `NUM_DATA_POINTS = 0` against a fixed positive count. Sparse observations are sampled only from the training-time interior; validation and test labels remain held out.

In [ ]:
!pip install -q wandb accelerate

In [ ]:
import os

base_path = '/kaggle/working'
project_name = 'weight-vs-loss-convergence-pinn'
project_root = os.path.join(base_path, project_name)

if not os.path.exists(project_root):
    os.chdir(base_path)
    !git clone https://github.com/nthday-jpg/weight-vs-loss-convergence-pinn.git
else:
    os.chdir(project_root)
    !git pull

os.environ['PYTHONPATH'] = project_root
%cd {project_root}

from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_KEY')

In [ ]:
# Change NUM_DATA_POINTS between runs; keep all other settings fixed.
FLOW_CASE = 'steep'        # smooth, steep, or shock
BALANCER = 'uniform'       # hold fixed for the data ablation
NUM_DATA_POINTS = 0        # compare 0 against, for example, 2048
SEED = 42

case_presets = {
    'smooth': {
        'nx': 256, 'steps': 200, 't_final': 0.6,
        'mu': 1e-2, 'spatial_scheme': 'central',
    },
    'steep': {
        'nx': 384, 'steps': 250, 't_final': 0.4,
        'mu': 3e-3, 'spatial_scheme': 'rusanov',
    },
    'shock': {
        'nx': 512, 'steps': 300, 't_final': 0.3,
        'mu': 5e-4, 'spatial_scheme': 'rusanov',
    },
}
assert FLOW_CASE in case_presets
assert BALANCER in {'uniform', 'softadapt'}
case = case_presets[FLOW_CASE]
base_name = f'navier_stokes_{FLOW_CASE}'
data_file = os.path.join(project_root, 'navier-stokes', 'data', base_name + '.pt')

import subprocess
import sys
subprocess.run([
    sys.executable, 'navier-stokes/data/gen_data.py',
    '--save_dir', 'navier-stokes/data',
    '--base_name', base_name,
    '--flow_case', FLOW_CASE,
    '--spatial_scheme', case['spatial_scheme'],
    '--nx', str(case['nx']),
    '--steps', str(case['steps']),
    '--t_final', str(case['t_final']),
    '--mu', str(case['mu']),
], check=True)

In [ ]:
import json

# One configuration produces one W&B run. Keep the architecture shared:
# the three outputs are coupled primitive variables (rho, u, p).
config = {
    'problem': 'navier_stokes',
    'layers': [2, 64, 64, 64, 3],
    'learning_rate': 1e-3,
    'num_epochs': 500,
    'step_per_epoch': 100,
    'batch_size': 2048,
    'seed': SEED,
    'train_time_fraction': 0.7,
    'validation_time_fraction': 0.85,
    'num_data_points': NUM_DATA_POINTS,
    'balancer_type': BALANCER,
    'softadapt_params': {
        'beta': 0.1,
        'history_length': 5,  # epoch means stored before adapting
        'normalized': False,
        'loss_weighted': False,
    },
    'wandb_project': 'loss-weight-convergence',
    'wandb_run_name': f'navier-stokes-{FLOW_CASE}-{BALANCER}-data-{NUM_DATA_POINTS}-seed-{SEED}',
}

# Environment variables are inherited by the Accelerate subprocess.
os.environ['PINN_CONFIG'] = json.dumps(config)
os.environ['PINN_DATA_FILE'] = data_file

In [ ]:
# One process keeps adaptive balancer state deterministic and synchronized.
!accelerate launch --num_processes 1 {project_root}/train.py

In [ ]:
from IPython.display import Image, display
plot_name = config['wandb_run_name'] + '_comparison.png'
display(Image(filename=os.path.join(project_root, 'results', plot_name)))
print('Convergence tables and time/step plots are available in the W&B run.')